# Production planning: linear programming with duality

A factory produces two products at profits $c = (40, 30)$ per unit, subject
to labour and material constraints. Solve the primal with `linprog`, extract
the dual from the solver output, and verify complementary slackness.

In [1]:
import numpy as np
from scipy.optimize import linprog

np.random.seed(0)

## 1. Problem statement

**Primal** (maximize, written as a minimise after sign flip):

$$\max\; 40 x_1 + 30 x_2 \quad \text{s.t.} \quad
  \begin{cases}
     x_1 + 2 x_2 \leq 40  & (\text{labour hours}) \\
     3 x_1 + 2 x_2 \leq 60 & (\text{raw material kg}) \\
     x_1, x_2 \geq 0.
  \end{cases}$$

**Dual**:

$$\min\; 40 y_1 + 60 y_2 \quad \text{s.t.} \quad
  \begin{cases}
    y_1 + 3 y_2 \geq 40 \\
    2 y_1 + 2 y_2 \geq 30 \\
    y_1, y_2 \geq 0.
  \end{cases}$$

The $y_i$ are the shadow prices of the two resources.

In [2]:
# linprog MINIMISES c @ x, so we pass -c to get a maximisation.
c = np.array([40.0, 30.0])
A_ub = np.array([[1.0, 2.0],
                 [3.0, 2.0]])
b_ub = np.array([40.0, 60.0])

res = linprog(c=-c, A_ub=A_ub, b_ub=b_ub,
              bounds=[(0, None), (0, None)], method="highs")
assert res.success, res.message

x_star = res.x
primal_value = c @ x_star
print(f"Primal optimum  x* = {x_star}")
print(f"Primal objective     = {primal_value:.4f}")

Primal optimum  x* = [10. 15.]
Primal objective     = 850.0000


## 2. Extract the dual from the solver output

HiGHS returns Lagrange multipliers in `res.ineqlin.marginals`. Signs follow
the SciPy convention where $A_{ub} x \leq b_{ub}$: the marginal is the
sensitivity of the (minimisation) objective to a relaxation of the
constraint, so for our maximisation we flip the sign.

In [3]:
y_star = -res.ineqlin.marginals
dual_value = b_ub @ y_star
print(f"Dual optimum   y* = {y_star}")
print(f"Dual objective       = {dual_value:.4f}")
print(f"Duality gap           = {abs(primal_value - dual_value):.2e}")

Dual optimum   y* = [ 2.5 12.5]
Dual objective       = 850.0000
Duality gap           = 0.00e+00


Strong duality: the two objectives match to machine precision.

## 3. Verify complementary slackness

At optimum,
$y_i^* \cdot (b_i - (A_{ub} x^*)_i) = 0$ for each primal constraint $i$,
and $x_j^* \cdot ((A_{ub}^\top y^*)_j - c_j) = 0$ for each primal variable $j$.

In [4]:
primal_slack = b_ub - A_ub @ x_star
dual_slack   = A_ub.T @ y_star - c

print("Primal slack  (b - A x*):        ", primal_slack)
print("Dual   slack  (A^T y* - c):      ", dual_slack)
print()
print("y_i * primal_slack_i (should be ~0):")
for i in range(2):
    print(f"  constraint {i+1}: y{i+1} * slack{i+1} = {y_star[i]} * {primal_slack[i]:.4f} = {y_star[i]*primal_slack[i]:.3e}")
print()
print("x_j * dual_slack_j (should be ~0):")
for j in range(2):
    print(f"  variable  {j+1}: x{j+1} * dualslack{j+1} = {x_star[j]:.4f} * {dual_slack[j]:.4f} = {x_star[j]*dual_slack[j]:.3e}")

Primal slack  (b - A x*):         [0. 0.]
Dual   slack  (A^T y* - c):       [0. 0.]

y_i * primal_slack_i (should be ~0):
  constraint 1: y1 * slack1 = 2.5 * 0.0000 = 0.000e+00
  constraint 2: y2 * slack2 = 12.5 * 0.0000 = 0.000e+00

x_j * dual_slack_j (should be ~0):
  variable  1: x1 * dualslack1 = 10.0000 * 0.0000 = 0.000e+00
  variable  2: x2 * dualslack2 = 15.0000 * 0.0000 = 0.000e+00


## 4. Interpret the shadow prices

$y_1^*$ is the marginal profit gain per extra hour of labour, and $y_2^*$
is the marginal profit gain per extra kg of raw material — the "price" the
factory would be willing to pay for one more unit of each resource.

In [5]:
print(f"Shadow price of labour hour:    y1* = {y_star[0]:.3f}")
print(f"Shadow price of raw material:   y2* = {y_star[1]:.3f}")

# Sanity check: perturb b1 by a small delta and re-solve, compare slope.
delta = 0.01
b_pert = b_ub + np.array([delta, 0.0])
res_pert = linprog(c=-c, A_ub=A_ub, b_ub=b_pert,
                   bounds=[(0, None), (0, None)], method="highs")
slope = (c @ res_pert.x - primal_value) / delta
print(f"Numerical dObj / db1 at optimum  = {slope:.3f}  (matches y1*)")

Shadow price of labour hour:    y1* = 2.500
Shadow price of raw material:   y2* = 12.500
Numerical dObj / db1 at optimum  = 2.500  (matches y1*)


## Exercises

1. Change $c$ to $(40, 50)$ (product 2 is now more profitable). Which
   constraint becomes binding? Which shadow price changes?
2. Add a third constraint $x_1 \leq 15$ (a production-line cap on product 1)
   and interpret its shadow price.
3. Dualise by hand on paper, then verify numerically that the dual's
   shadow prices on *its* constraints recover the primal $x^*$ (this is
   the "the dual of the dual is the primal" identity).